# ETH Price Forecasting with Graph Attention

This notebook is a full rewrite focused on one primary goal: forecasting the future ETH return for a fixed horizon using the same market data and asset universe as before.

Key design choices:
- Primary objective is return regression for ETH (not trade classification).
- RMSE and MAE are tracked as intermediate optimization metrics.
- Trading PnL is evaluated from model forecasts using out-of-sample threshold selection.
- Walk-forward splits use purge/embargo gaps to reduce label leakage.
- Final report includes holdout test PnL from a production-fit model.

In [1]:
import json
import math
import os
import random
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.preprocessing import RobustScaler
from torch.utils.data import DataLoader, Dataset


def seed_everything(seed: int = 1234) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(100)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

torch.set_num_threads(max(1, os.cpu_count() or 4))

CFG: Dict[str, Any] = {
    # data
    "freq": "1min",
    "data_dir": "../dataset",
    "data_slice_start_frac": 0.00,
    "data_slice_end_frac": 0.75,
    "final_test_frac": 0.10,
    # order book
    "book_levels": 15,
    "top_levels": 5,
    "near_levels": 5,
    # walk-forward windows
    "train_min_frac": 0.55,
    "val_window_frac": 0.10,
    "test_window_frac": 0.10,
    "step_window_frac": 0.10,
    # scaling
    "max_abs_feat": 6.0,
    "max_abs_edge": 3.5,
    # edge features
    "corr_windows": [10, 30, 60, 120],
    "corr_lags": [0, 1, 2, 5, 10, 30],
    "edges_mode": "all_pairs",
    "add_self_loops": True,
    "edge_transform": "fisher",
    "edge_scale": True,
    # labels and horizons
    "tb_horizon": 30,
    "lookback": 240,
    "tb_pt_mult": 1.70,
    "tb_sl_mult": 1.70,
    "tb_min_barrier": 0.0035,
    "tb_max_barrier": 0.0150,
    "fixed_horizon": 30,
    "fixed_ret_clip": 0.010,
    # training
    "batch_size": 64,
    "epochs": 50,
    "lr": 2.5e-4,
    "weight_decay": 4.0e-3,
    "grad_clip": 1.0,
    "dropout": 0.20,
    "use_onecycle": True,
    "onecycle_pct_start": 0.20,
    "onecycle_div_factor": 30.0,
    "onecycle_final_div": 600.0,
    # temporal encoder
    "attn_model_dim": 64,
    "attn_num_heads": 4,
    "attn_num_layers": 3,
    "attn_ff_mult": 4,
    # graph module
    "graph_num_layers": 2,
    # adaptive adjacency
    "adj_emb_dim": 16,
    "adj_temperature": 1.25,
    "adaptive_topk": 3,
    "adj_l1_lambda": 2e-3,
    "adj_prior_lambda": 1e-2,
    "prior_diag_boost": 1.0,
    "prior_row_normalize": True,
    # regression
    "ret_huber_delta": 0.0015,
    # trading evaluation
    "cost_bps": 1.0,
    "signal_thr_grid": [0.0001, 0.0002, 0.0003, 0.0005, 0.0007, 0.0010, 0.0015],
    "eval_min_trades": 100,
    "max_trade_rate_val": 0.06,
    "trade_rate_penalty": 6.0,
    "pnl_objective": "pnl_sum",
    "proxy_target_trades": [100, 200, 400, 800],
    # artifact
    "artifact_dir": "./artifacts_attention_1m",
}

ASSETS = ["ADA", "BTC", "ETH"]
ASSET2IDX = {a: i for i, a in enumerate(ASSETS)}
TARGET_ASSET = "ETH"
TARGET_NODE = ASSET2IDX[TARGET_ASSET]

ART_DIR = Path(CFG["artifact_dir"])
ART_DIR.mkdir(parents=True, exist_ok=True)

print("Assets:", ASSETS, "| Target:", TARGET_ASSET)
print("Artifacts dir:", str(ART_DIR.resolve()))

DEVICE: cpu
Assets: ['ADA', 'BTC', 'ETH'] | Target: ETH
Artifacts dir: /Users/vitalii/Desktop/Model_Market_Microstructure/Graph_Neural_Network_for_Market_Microstructure/TGNN_1sec/artifacts_attention_1m


# Graph Topology

Build a directed graph over the same three assets. The graph is reused by both edge-feature construction and message passing.

In [2]:
def build_edge_list(cfg: Dict[str, Any], assets: List[str]) -> List[Tuple[str, str]]:
    mode = str(cfg.get("edges_mode", "manual"))
    if mode == "manual":
        edges = list(cfg["edges"])
    elif mode == "all_pairs":
        edges = [(s, t) for s in assets for t in assets if s != t]
    else:
        raise ValueError(f"Unknown edges_mode={mode}")

    if bool(cfg.get("add_self_loops", True)):
        edges = edges + [(a, a) for a in assets]
    return edges


EDGE_LIST = build_edge_list(CFG, ASSETS)
EDGE_NAMES = [f"{s}->{t}" for (s, t) in EDGE_LIST]
EDGE_INDEX = torch.tensor([[ASSET2IDX[s], ASSET2IDX[t]] for (s, t) in EDGE_LIST], dtype=torch.long)

print("EDGE_LIST:", EDGE_NAMES)
print("EDGE_INDEX:", EDGE_INDEX.tolist())

EDGE_LIST: ['ADA->BTC', 'ADA->ETH', 'BTC->ADA', 'BTC->ETH', 'ETH->ADA', 'ETH->BTC', 'ADA->ADA', 'BTC->BTC', 'ETH->ETH']
EDGE_INDEX: [[0, 1], [0, 2], [1, 0], [1, 2], [2, 0], [2, 1], [0, 0], [1, 1], [2, 2]]


# Data Loading

Load the same ADA/BTC/ETH files, align by second-level timestamp, and construct per-asset log returns.

In [3]:
def load_asset(
    asset: str,
    freq: str,
    data_dir: Path,
    book_levels: int,
    part: Tuple[float, float],
) -> pd.DataFrame:
    path = data_dir / f"{asset}_{freq}.csv"
    df = pd.read_csv(path)

    start = int(len(df) * float(part[0]))
    end = int(len(df) * float(part[1]))
    df = df.iloc[start:end]

    df["timestamp"] = pd.to_datetime(df["system_time"]).dt.round("s")
    df = df.sort_values("timestamp").set_index("timestamp")

    bid_cols = [f"bids_notional_{i}" for i in range(book_levels)]
    ask_cols = [f"asks_notional_{i}" for i in range(book_levels)]
    needed = ["midpoint", "spread", "buys", "sells"] + bid_cols + ask_cols

    missing = [c for c in needed if c not in df.columns]
    if missing:
        raise ValueError(f"{asset}: missing columns: {missing[:10]}")

    return df[needed]


def load_all_assets() -> pd.DataFrame:
    freq = CFG["freq"]
    data_dir = Path(CFG["data_dir"])
    book_levels = int(CFG["book_levels"])
    part = (float(CFG["data_slice_start_frac"]), float(CFG["data_slice_end_frac"]))

    def rename_cols(df_one: pd.DataFrame, asset: str) -> pd.DataFrame:
        rename_map = {
            "midpoint": asset,
            "buys": f"buys_{asset}",
            "sells": f"sells_{asset}",
            "spread": f"spread_{asset}",
        }
        for i in range(book_levels):
            rename_map[f"bids_notional_{i}"] = f"bids_vol_{asset}_{i}"
            rename_map[f"asks_notional_{i}"] = f"asks_vol_{asset}_{i}"
        return df_one.rename(columns=rename_map)

    df_ada = rename_cols(load_asset("ADA", freq, data_dir, book_levels, part=part), "ADA")
    df_btc = rename_cols(load_asset("BTC", freq, data_dir, book_levels, part=part), "BTC")
    df_eth = rename_cols(load_asset("ETH", freq, data_dir, book_levels, part=part), "ETH")

    out = df_ada.join(df_btc, how="inner").join(df_eth, how="inner").reset_index()
    for a in ASSETS:
        out[f"lr_{a}"] = np.log(out[a]).diff().fillna(0.0)
    return out


df = load_all_assets()
print("Loaded df:", df.shape)
print("Time range:", df["timestamp"].min(), "->", df["timestamp"].max())
print(df.head(2))

Loaded df: (8261, 106)
Time range: 2021-04-08 00:01:00+00:00 -> 2021-04-16 10:15:00+00:00
                  timestamp      ADA  spread_ADA      buys_ADA     sells_ADA  \
0 2021-04-08 00:01:00+00:00  1.17395      0.0003  34639.037982  53033.253320   
1 2021-04-08 00:02:00+00:00  1.17335      0.0007  13553.749385  15412.718329   

   bids_vol_ADA_0  bids_vol_ADA_1  bids_vol_ADA_2  bids_vol_ADA_3  \
0     3051.879883     1071.420044    10819.780273      816.969971   
1     3049.800049      586.450012     1071.420044     4377.330078   

   bids_vol_ADA_4  ...  asks_vol_ETH_8  asks_vol_ETH_9  asks_vol_ETH_10  \
0       16.440001  ...      984.049988    77191.632812       738.070007   
1     2701.510010  ...       35.389999    20013.490234     20013.619141   

   asks_vol_ETH_11  asks_vol_ETH_12  asks_vol_ETH_13  asks_vol_ETH_14  \
0     23421.820312      7812.069824       829.179993        709.97998   
1       551.979980      3943.989990        88.599998      13125.75000   

     lr_ADA    

# Edge Features

Create rolling lead-lag correlation features for each directed edge and each time step.

In [4]:
def _fisher_z(x: np.ndarray, eps: float = 1e-6) -> np.ndarray:
    x = np.clip(x, -0.999, 0.999)
    return 0.5 * np.log((1.0 + x + eps) / (1.0 - x + eps))


def build_corr_array(
    df_: pd.DataFrame,
    corr_windows: List[int],
    edges: List[Tuple[str, str]],
    lags: List[int],
    transform: str = "fisher",
) -> np.ndarray:
    t_len = len(df_)
    e_len = len(edges)
    w_len = len(corr_windows)
    l_len = len(lags)
    out = np.zeros((t_len, e_len, w_len * l_len), dtype=np.float32)

    lr_map = {a: df_[f"lr_{a}"].astype(float) for a in ASSETS}

    for ei, (s, t) in enumerate(edges):
        if s == t:
            out[:, ei, :] = 1.0
            continue

        src0 = lr_map[s]
        dst0 = lr_map[t]

        feat_idx = 0
        for lag in lags:
            src = src0.shift(int(lag)) if int(lag) > 0 else src0
            for w in corr_windows:
                r = src.rolling(int(w), min_periods=1).corr(dst0)
                r = np.nan_to_num(r.to_numpy(dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)
                if transform == "fisher":
                    r = _fisher_z(r).astype(np.float32)
                out[:, ei, feat_idx] = r
                feat_idx += 1

    return out.astype(np.float32)


edge_feat = build_corr_array(
    df,
    CFG["corr_windows"],
    EDGE_LIST,
    CFG["corr_lags"],
    transform=str(CFG.get("edge_transform", "fisher")),
)

print("edge_feat shape:", edge_feat.shape, "(T,E,D)")

edge_feat shape: (8261, 9, 24) (T,E,D)


# Targets

Primary target: fixed-horizon ETH return for regression.

Auxiliary evaluation target: triple-barrier realized return for trading PnL backtest.

In [5]:
EPS = 1e-6


def triple_barrier_exit_return_from_lr(
    lr: pd.Series,
    horizon: int,
    vol_window: int,
    pt_mult: float,
    sl_mult: float,
    min_barrier: float,
    max_barrier: float,
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    lr = lr.astype(float).copy()
    t_len = len(lr)

    vol = lr.rolling(vol_window, min_periods=max(10, vol_window // 10)).std().shift(1)
    thr = (vol * np.sqrt(horizon)).clip(lower=min_barrier, upper=max_barrier)

    exit_ret = np.zeros(t_len, dtype=np.float32)
    exit_t = np.arange(t_len, dtype=np.int64)

    lr_np = lr.fillna(0.0).to_numpy(dtype=np.float64)
    thr_np = thr.fillna(min_barrier).to_numpy(dtype=np.float64)

    for t in range(t_len - horizon - 1):
        up = pt_mult * thr_np[t]
        dn = -sl_mult * thr_np[t]

        cum = 0.0
        et = t + horizon
        er = 0.0

        for dt in range(1, horizon + 1):
            cum += lr_np[t + dt]
            if cum >= up or cum <= dn:
                et, er = t + dt, cum
                break

        if et == t + horizon:
            er = float(np.sum(lr_np[t + 1: t + horizon + 1]))

        exit_ret[t] = er
        exit_t[t] = et

    return exit_ret, exit_t, thr_np


def fixed_horizon_future_return(lr: np.ndarray, horizon: int) -> np.ndarray:
    lr = np.asarray(lr, dtype=np.float64)
    t_len = lr.shape[0]
    out = np.zeros(t_len, dtype=np.float32)
    if horizon <= 0:
        return out
    for t in range(0, t_len - horizon - 1):
        out[t] = float(lr[t + 1: t + horizon + 1].sum())
    return out


exit_ret, exit_t, tb_thr = triple_barrier_exit_return_from_lr(
    df["lr_ETH"],
    horizon=int(CFG["tb_horizon"]),
    vol_window=int(CFG["lookback"]),
    pt_mult=float(CFG["tb_pt_mult"]),
    sl_mult=float(CFG["tb_sl_mult"]),
    min_barrier=float(CFG["tb_min_barrier"]),
    max_barrier=float(CFG["tb_max_barrier"]),
)

ret_target = fixed_horizon_future_return(df["lr_ETH"].to_numpy(dtype=np.float64), int(CFG["fixed_horizon"]))

print("ret_target mean:", float(np.mean(ret_target)), "std:", float(np.std(ret_target)))
print("exit_ret mean:", float(np.mean(exit_ret)), "std:", float(np.std(exit_ret)))

ret_target mean: 0.0007000209880061448 std: 0.007016943767666817
exit_ret mean: 0.0005145660252310336 std: 0.006837026681751013


# Node Features and Sample Index Range

Build node features for all assets and define valid sample timestamps for lookback and future horizon constraints.

In [6]:
def safe_log1p(x: np.ndarray) -> np.ndarray:
    return np.log1p(np.maximum(x, 0.0))


def build_node_tensor(df_: pd.DataFrame) -> Tuple[np.ndarray, List[str]]:
    book_levels = int(CFG["book_levels"])
    top_k = int(CFG["top_levels"])
    near_k = int(CFG["near_levels"])

    if near_k >= book_levels:
        raise ValueError("CFG['near_levels'] must be < CFG['book_levels']")

    feat_names = [
        "lr", "spread", "log_buys", "log_sells", "ofi", "DI_15",
        "DI_L0", "DI_L1", "DI_L2", "DI_L3", "DI_L4",
        "near_ratio_bid", "near_ratio_ask", "di_near", "di_far",
    ]

    feats_all = []
    for a in ASSETS:
        lr = df_[f"lr_{a}"].values.astype(np.float32)
        spread = df_[f"spread_{a}"].values.astype(np.float32)

        buys = df_[f"buys_{a}"].values.astype(np.float32)
        sells = df_[f"sells_{a}"].values.astype(np.float32)

        log_buys = safe_log1p(buys).astype(np.float32)
        log_sells = safe_log1p(sells).astype(np.float32)
        ofi = ((buys - sells) / (buys + sells + EPS)).astype(np.float32)

        bids_lvls = np.stack([df_[f"bids_vol_{a}_{i}"].values.astype(np.float32) for i in range(book_levels)], axis=1)
        asks_lvls = np.stack([df_[f"asks_vol_{a}_{i}"].values.astype(np.float32) for i in range(book_levels)], axis=1)

        bid_sum = bids_lvls.sum(axis=1)
        ask_sum = asks_lvls.sum(axis=1)
        di_15 = ((bid_sum - ask_sum) / (bid_sum + ask_sum + EPS)).astype(np.float32)

        di_levels = []
        for i in range(top_k):
            b = bids_lvls[:, i]
            s = asks_lvls[:, i]
            di_levels.append(((b - s) / (b + s + EPS)).astype(np.float32))
        di_l0_4 = np.stack(di_levels, axis=1)

        bid_near = bids_lvls[:, :near_k].sum(axis=1)
        ask_near = asks_lvls[:, :near_k].sum(axis=1)
        bid_far = bids_lvls[:, near_k:].sum(axis=1)
        ask_far = asks_lvls[:, near_k:].sum(axis=1)

        near_ratio_bid = (bid_near / (bid_far + EPS)).astype(np.float32)
        near_ratio_ask = (ask_near / (ask_far + EPS)).astype(np.float32)
        di_near = ((bid_near - ask_near) / (bid_near + ask_near + EPS)).astype(np.float32)
        di_far = ((bid_far - ask_far) / (bid_far + ask_far + EPS)).astype(np.float32)

        xa = np.column_stack([
            lr, spread, log_buys, log_sells, ofi, di_15,
            di_l0_4[:, 0], di_l0_4[:, 1], di_l0_4[:, 2], di_l0_4[:, 3], di_l0_4[:, 4],
            near_ratio_bid, near_ratio_ask, di_near, di_far,
        ]).astype(np.float32)

        feats_all.append(xa)

    x = np.stack(feats_all, axis=1).astype(np.float32)
    return x, feat_names


X_node_raw, node_feat_names = build_node_tensor(df)

T = len(df)
L = int(CFG["lookback"])
H_TB = int(CFG["tb_horizon"])
H_FIXED = int(CFG["fixed_horizon"])

# Keep only t where lookback exists and future targets are valid.
t_min = L - 1
t_max = T - max(H_TB, H_FIXED) - 2
sample_t = np.arange(t_min, t_max + 1)
n_samples = len(sample_t)

print("X_node_raw:", X_node_raw.shape, "edge_feat:", edge_feat.shape)
print("n_samples:", n_samples, "| t range:", int(sample_t[0]), "->", int(sample_t[-1]))

X_node_raw: (8261, 3, 15) edge_feat: (8261, 9, 24)
n_samples: 7991 | t range: 239 -> 8229


# Splits with Purge/Embargo

Use walk-forward splits with a safety gap between train/validation/test windows.
The gap is set to the maximum forecast horizon to reduce future-label leakage.

In [7]:
def make_final_holdout_split(n_samples_: int, final_test_frac: float) -> Tuple[np.ndarray, np.ndarray]:
    if not (0.0 < final_test_frac < 0.5):
        raise ValueError("final_test_frac should be in (0, 0.5)")
    n_final = max(1, int(round(final_test_frac * n_samples_)))
    n_cv = n_samples_ - n_final
    if n_cv <= 50:
        raise ValueError("Too few samples left for CV after holdout split.")
    idx_cv = np.arange(0, n_cv, dtype=np.int64)
    idx_final = np.arange(n_cv, n_samples_, dtype=np.int64)
    return idx_cv, idx_final


def make_walk_forward_splits_with_gap(
    n_samples_: int,
    train_min_frac: float,
    val_window_frac: float,
    test_window_frac: float,
    step_window_frac: float,
    gap: int,
) -> List[Tuple[np.ndarray, np.ndarray, np.ndarray]]:
    train_min = int(train_min_frac * n_samples_)
    val_w = max(1, int(val_window_frac * n_samples_))
    test_w = max(1, int(test_window_frac * n_samples_))
    step_w = max(1, int(step_window_frac * n_samples_))

    splits: List[Tuple[np.ndarray, np.ndarray, np.ndarray]] = []
    start = train_min

    while True:
        tr_end = start
        va_start = tr_end + gap
        va_end = va_start + val_w
        te_start = va_end + gap
        te_end = te_start + test_w

        if te_end > n_samples_:
            break

        idx_train = np.arange(0, tr_end, dtype=np.int64)
        idx_val = np.arange(va_start, va_end, dtype=np.int64)
        idx_test = np.arange(te_start, te_end, dtype=np.int64)

        if len(idx_train) > 0 and len(idx_val) > 0 and len(idx_test) > 0:
            splits.append((idx_train, idx_val, idx_test))

        start += step_w

    return splits


idx_cv_all, idx_final_test = make_final_holdout_split(n_samples, float(CFG["final_test_frac"]))
n_samples_cv = len(idx_cv_all)

purge_gap = max(int(CFG["tb_horizon"]), int(CFG["fixed_horizon"]))
walk_splits = make_walk_forward_splits_with_gap(
    n_samples_=n_samples_cv,
    train_min_frac=float(CFG["train_min_frac"]),
    val_window_frac=float(CFG["val_window_frac"]),
    test_window_frac=float(CFG["test_window_frac"]),
    step_window_frac=float(CFG["step_window_frac"]),
    gap=purge_gap,
)

print("Holdout split:")
print(f"  n_samples total: {n_samples}")
print(f"  n_samples CV   : {len(idx_cv_all)}")
print(f"  n_samples FINAL: {len(idx_final_test)}")
print("Purge/embargo gap:", purge_gap)
print("\nWalk-forward folds:", len(walk_splits))
for i, (a, b, c) in enumerate(walk_splits, 1):
    print(f"  fold {i}: train={len(a)} | val={len(b)} | test={len(c)}")

Holdout split:
  n_samples total: 7991
  n_samples CV   : 7192
  n_samples FINAL: 799
Purge/embargo gap: 30

Walk-forward folds: 3
  fold 1: train=3955 | val=719 | test=719
  fold 2: train=4674 | val=719 | test=719
  fold 3: train=5393 | val=719 | test=719


# Dataset, Scaling, Model, and Evaluation Utilities

This block defines:
- sequence dataset for ETH return regression,
- train-only robust scaling,
- attention + graph model,
- RMSE/MAE metrics,
- PnL utilities from signed forecast signals.

In [8]:
class LobGraphSequenceDatasetPrice(Dataset):
    def __init__(
        self,
        x_node: np.ndarray,
        e_feat: np.ndarray,
        y_ret_arr: np.ndarray,
        exit_ret_arr: np.ndarray,
        sample_t_: np.ndarray,
        indices: np.ndarray,
        lookback: int,
    ):
        self.x_node = x_node
        self.e_feat = e_feat
        self.y_ret = y_ret_arr
        self.exit_ret = exit_ret_arr
        self.sample_t = sample_t_
        self.indices = indices.astype(np.int64)
        self.lookback = int(lookback)

    def __len__(self) -> int:
        return int(len(self.indices))

    def __getitem__(self, i: int):
        sidx = int(self.indices[i])
        t = int(self.sample_t[sidx])
        t0 = t - self.lookback + 1

        x_seq = self.x_node[t0: t + 1]
        e_last = self.e_feat[t]
        y_ret = float(self.y_ret[t])
        er_exit = float(self.exit_ret[t])

        return (
            torch.from_numpy(x_seq),
            torch.from_numpy(e_last),
            torch.tensor(y_ret, dtype=torch.float32),
            torch.tensor(er_exit, dtype=torch.float32),
            torch.tensor(sidx, dtype=torch.long),
        )


def collate_fn_price(batch):
    xs, e_last, yret, er_exit, sidxs = zip(*batch)
    return (
        torch.stack(xs, 0),
        torch.stack(e_last, 0),
        torch.stack(yret, 0),
        torch.stack(er_exit, 0),
        torch.stack(sidxs, 0),
    )


def fit_scale_nodes_train_only(
    x_node_raw_: np.ndarray,
    sample_t_: np.ndarray,
    idx_train: np.ndarray,
    max_abs: float,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    last_train_t = int(sample_t_[int(idx_train[-1])])
    train_time_mask = np.arange(0, last_train_t + 1)

    x_train_time = x_node_raw_[train_time_mask]
    _, _, f_dim = x_train_time.shape

    scaler = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(5.0, 95.0))
    scaler.fit(x_train_time.reshape(-1, f_dim))

    x_scaled = scaler.transform(x_node_raw_.reshape(-1, f_dim)).reshape(x_node_raw_.shape).astype(np.float32)
    x_scaled = np.clip(x_scaled, -max_abs, max_abs).astype(np.float32)
    x_scaled = np.nan_to_num(x_scaled, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    params = {
        "center_": scaler.center_.astype(np.float32),
        "scale_": scaler.scale_.astype(np.float32),
        "max_abs": float(max_abs),
    }
    return x_scaled, params


def fit_scale_edges_train_only(
    e_raw_: np.ndarray,
    sample_t_: np.ndarray,
    idx_train: np.ndarray,
    max_abs: float,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    last_train_t = int(sample_t_[int(idx_train[-1])])
    train_time_mask = np.arange(0, last_train_t + 1)

    e_train_time = e_raw_[train_time_mask]
    _, _, d_dim = e_train_time.shape

    scaler = RobustScaler(with_centering=True, with_scaling=True, quantile_range=(5.0, 95.0))
    scaler.fit(e_train_time.reshape(-1, d_dim))

    e_scaled = scaler.transform(e_raw_.reshape(-1, d_dim)).reshape(e_raw_.shape).astype(np.float32)
    e_scaled = np.clip(e_scaled, -max_abs, max_abs).astype(np.float32)
    e_scaled = np.nan_to_num(e_scaled, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

    params = {
        "center_": scaler.center_.astype(np.float32),
        "scale_": scaler.scale_.astype(np.float32),
        "max_abs": float(max_abs),
    }
    return e_scaled, params


def apply_scaler_params(x: np.ndarray, params: Dict[str, Any]) -> np.ndarray:
    center = np.asarray(params["center_"], dtype=np.float32)
    scale = np.asarray(params["scale_"], dtype=np.float32)
    max_abs = float(params["max_abs"])
    x2 = (x - center) / (scale + 1e-12)
    x2 = np.clip(x2, -max_abs, max_abs)
    return np.nan_to_num(x2, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def build_static_adjacency_from_edges(edge_index: torch.Tensor, n_nodes: int, eps: float = 1e-8) -> torch.Tensor:
    a = torch.zeros((n_nodes, n_nodes), dtype=torch.float32)
    src = edge_index[:, 0].long()
    dst = edge_index[:, 1].long()
    a[src, dst] = 1.0
    a = a / (a.sum(dim=-1, keepdim=True) + eps)
    return a


def build_adj_prior_from_edge_attr(
    edge_attr_last: torch.Tensor,
    edge_index: torch.Tensor,
    n_nodes: int,
    diag_boost: float,
    row_normalize: bool,
    eps: float = 1e-8,
) -> torch.Tensor:
    edge_attr_last = torch.nan_to_num(edge_attr_last, nan=0.0, posinf=0.0, neginf=0.0)
    bsz, _, _ = edge_attr_last.shape

    # Keep signed information with tanh, then map to [0, 1].
    signed_strength = torch.tanh(edge_attr_last.mean(dim=-1))
    w = 0.5 * (signed_strength + 1.0)

    a = torch.zeros((bsz, n_nodes, n_nodes), device=edge_attr_last.device, dtype=edge_attr_last.dtype)
    src = edge_index[:, 0].to(edge_attr_last.device)
    dst = edge_index[:, 1].to(edge_attr_last.device)
    a[:, src, dst] = w

    diag = torch.arange(n_nodes, device=edge_attr_last.device)
    a[:, diag, diag] = torch.maximum(a[:, diag, diag], torch.full_like(a[:, diag, diag], float(diag_boost)))

    if row_normalize:
        a = a / (a.sum(dim=-1, keepdim=True) + eps)

    return torch.nan_to_num(a, nan=0.0, posinf=0.0, neginf=0.0)


class AdaptiveAdjacency(nn.Module):
    def __init__(self, n_nodes: int, cfg: Dict[str, Any]):
        super().__init__()
        self.n = int(n_nodes)
        k = int(cfg.get("adj_emb_dim", 8))
        self.e1 = nn.Parameter(0.01 * torch.randn(self.n, k))
        self.e2 = nn.Parameter(0.01 * torch.randn(self.n, k))
        temp = float(cfg.get("adj_temperature", 1.0))
        self.temp = max(temp, 1e-3)
        self.topk = int(cfg.get("adaptive_topk", self.n))

    def forward(self) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        logits = (self.e1 @ self.e2.t()) / self.temp

        if 0 < self.topk < self.n:
            vals, idx = torch.topk(logits.abs(), k=self.topk, dim=-1)
            masked = torch.full_like(logits, fill_value=float("-inf"))
            masked.scatter_(-1, idx, logits.gather(-1, idx))
            logits = masked

        a = torch.softmax(logits, dim=-1)
        sparsity_proxy = torch.sigmoid(torch.nan_to_num(logits, nan=0.0, neginf=-30.0, posinf=30.0))
        return a, sparsity_proxy, logits


class LearnableSupportMix(nn.Module):
    def __init__(self, n_supports: int = 3):
        super().__init__()
        self.w_logits = nn.Parameter(torch.zeros(n_supports, dtype=torch.float32))

    def forward(self) -> torch.Tensor:
        return torch.softmax(self.w_logits, dim=0)


class GraphPropagationBlock(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float):
        super().__init__()
        self.self_lin = nn.Linear(hidden_dim, hidden_dim)
        self.msg_lin = nn.Linear(hidden_dim, hidden_dim)
        self.norm = nn.LayerNorm(hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, h: torch.Tensor, a: torch.Tensor) -> torch.Tensor:
        msg = torch.einsum("bnm,bmd->bnd", a, h)
        update = self.self_lin(h) + self.msg_lin(msg)
        update = F.gelu(update)
        update = self.dropout(update)
        return self.norm(h + update)


class AttentionGraphPriceModel(nn.Module):
    def __init__(self, node_in: int, cfg: Dict[str, Any], n_nodes: int, target_node: int):
        super().__init__()
        self.cfg = cfg
        self.n_nodes = int(n_nodes)
        self.target_node = int(target_node)
        self.lookback = int(cfg["lookback"])

        model_dim = int(cfg["attn_model_dim"])
        heads = int(cfg["attn_num_heads"])
        layers = int(cfg["attn_num_layers"])
        ff_mult = int(cfg.get("attn_ff_mult", 4))
        dropout = float(cfg.get("dropout", 0.0))
        graph_layers = int(cfg.get("graph_num_layers", 2))

        self.node_proj = nn.Linear(int(node_in), model_dim)
        self.pos_emb = nn.Parameter(torch.zeros(1, self.lookback, 1, model_dim))

        enc_layer = nn.TransformerEncoderLayer(
            d_model=model_dim,
            nhead=heads,
            dim_feedforward=ff_mult * model_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.temporal_encoder = nn.TransformerEncoder(enc_layer, num_layers=layers)

        a_static = build_static_adjacency_from_edges(EDGE_INDEX, n_nodes=self.n_nodes)
        self.register_buffer("a_static", a_static)

        self.adapt = AdaptiveAdjacency(n_nodes=self.n_nodes, cfg=cfg)
        self.support_mix = LearnableSupportMix(n_supports=3)
        self.graph_blocks = nn.ModuleList([GraphPropagationBlock(model_dim, dropout) for _ in range(graph_layers)])

        self.end_norm = nn.LayerNorm(model_dim)
        self.ret_head = nn.Linear(model_dim, 1)

        nn.init.trunc_normal_(self.pos_emb, std=0.02)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None:
                    nn.init.zeros_(m.bias)

    def _compute_supports(self, e_last: torch.Tensor) -> Tuple[torch.Tensor, Dict[str, Any]]:
        e_last = torch.nan_to_num(e_last, nan=0.0, posinf=0.0, neginf=0.0)
        bsz, _, _ = e_last.shape

        a_prior = build_adj_prior_from_edge_attr(
            edge_attr_last=e_last,
            edge_index=EDGE_INDEX.to(e_last.device),
            n_nodes=self.n_nodes,
            diag_boost=float(self.cfg.get("prior_diag_boost", 1.0)),
            row_normalize=bool(self.cfg.get("prior_row_normalize", True)),
        )

        a_adapt_base, sparsity_proxy, _ = self.adapt()
        a_adapt = a_adapt_base.unsqueeze(0).expand(bsz, -1, -1)

        w = self.support_mix()
        a_static = self.a_static.to(e_last.device).to(e_last.dtype).unsqueeze(0).expand(bsz, -1, -1)
        a_mix = w[0] * a_static + w[1] * a_prior + w[2] * a_adapt
        a_mix = a_mix / (a_mix.sum(dim=-1, keepdim=True) + 1e-8)

        n = self.n_nodes
        offdiag = (1.0 - torch.eye(n, device=e_last.device, dtype=e_last.dtype))
        l1_off = (sparsity_proxy.to(e_last.dtype) * offdiag).abs().mean()
        mse_prior = ((a_adapt - a_prior) ** 2 * offdiag).mean()

        aux = {
            "support_w": w.detach().cpu().numpy().tolist(),
            "_l1_off_t": l1_off,
            "_mse_prior_t": mse_prior,
        }
        return a_mix, aux

    def forward(self, x_seq: torch.Tensor, e_last: torch.Tensor, return_aux: bool = False):
        x_seq = torch.nan_to_num(x_seq, nan=0.0, posinf=0.0, neginf=0.0)
        e_last = torch.nan_to_num(e_last, nan=0.0, posinf=0.0, neginf=0.0)

        bsz, seq_len, n_nodes, _ = x_seq.shape
        assert n_nodes == self.n_nodes
        if seq_len > self.lookback:
            raise ValueError(f"Sequence length {seq_len} exceeds configured lookback {self.lookback}")

        h = self.node_proj(x_seq)
        h = h + self.pos_emb[:, :seq_len]

        h = h.permute(0, 2, 1, 3).contiguous().view(bsz * n_nodes, seq_len, -1)
        h = self.temporal_encoder(h)
        h_last = h[:, -1, :].view(bsz, n_nodes, -1)

        a_mix, aux = self._compute_supports(e_last)
        for block in self.graph_blocks:
            h_last = block(h_last, a_mix)

        feat = self.end_norm(h_last[:, self.target_node, :])
        ret_hat = self.ret_head(feat).squeeze(-1)
        ret_hat = torch.nan_to_num(ret_hat, nan=0.0, posinf=0.0, neginf=0.0)

        if return_aux:
            return ret_hat, aux
        return ret_hat


def total_loss_with_adj_reg(loss: torch.Tensor, aux: Dict[str, Any], cfg: Dict[str, Any]) -> torch.Tensor:
    lam_l1 = float(cfg.get("adj_l1_lambda", 0.0))
    lam_pr = float(cfg.get("adj_prior_lambda", 0.0))
    reg = 0.0
    if lam_l1 > 0:
        reg = reg + lam_l1 * aux["_l1_off_t"]
    if lam_pr > 0:
        reg = reg + lam_pr * aux["_mse_prior_t"]
    return loss + reg


def regression_loss(ret_hat: torch.Tensor, ret_true: torch.Tensor, cfg: Dict[str, Any]) -> torch.Tensor:
    ret_hat = ret_hat.reshape(-1)
    ret_true = ret_true.reshape(-1)

    clip_val = float(cfg.get("fixed_ret_clip", 0.0))
    if clip_val > 0:
        ret_true = torch.clamp(ret_true, -clip_val, clip_val)

    delta = float(cfg.get("ret_huber_delta", 0.001))
    return F.huber_loss(ret_hat, ret_true, delta=delta, reduction="mean")


def rmse_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size == 0:
        return float("nan")
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae_np(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    if y_true.size == 0:
        return float("nan")
    return float(np.mean(np.abs(y_true - y_pred)))


def pnl_from_ret_forecast(
    pred_ret: np.ndarray,
    exit_ret_arr: np.ndarray,
    signal_thr: float,
    cost_bps: float,
) -> Dict[str, Any]:
    pred_ret = np.asarray(pred_ret, dtype=np.float64)
    exit_ret_arr = np.asarray(exit_ret_arr, dtype=np.float64)

    long_mask = pred_ret >= float(signal_thr)
    short_mask = pred_ret <= -float(signal_thr)
    trade_mask = long_mask | short_mask

    action = np.zeros_like(exit_ret_arr, dtype=np.float64)
    action[long_mask] = 1.0
    action[short_mask] = -1.0

    cost = (float(cost_bps) * 1e-4) * trade_mask.astype(np.float64)
    pnl = action * exit_ret_arr - cost

    n = int(len(exit_ret_arr))
    n_tr = int(trade_mask.sum())

    return {
        "n": n,
        "n_trades": n_tr,
        "trade_rate": float(n_tr / max(1, n)),
        "pnl_sum": float(pnl.sum()),
        "pnl_mean": float(pnl.mean()) if n else float("nan"),
        "pnl_per_trade": float(pnl.sum() / max(1, n_tr)),
    }


def build_signal_threshold_grid(
    pred_ret: np.ndarray,
    base_grid: Optional[List[float]],
    target_trades_list: Optional[List[int]],
) -> List[float]:
    pred_abs = np.abs(np.asarray(pred_ret, dtype=np.float64))
    pred_abs = pred_abs[np.isfinite(pred_abs)]
    if pred_abs.size == 0:
        return base_grid or [0.0]

    thrs = set(float(t) for t in (base_grid or []))

    if target_trades_list:
        n = int(pred_abs.size)
        for k in target_trades_list:
            k = int(k)
            if k <= 0:
                continue
            if k >= n:
                thr = float(np.min(pred_abs))
            else:
                q = 1.0 - (k / n)
                thr = float(np.quantile(pred_abs, q))
            thrs.add(max(0.0, thr))

    out = sorted(thrs)
    cleaned = []
    for t in out:
        if not cleaned or abs(t - cleaned[-1]) > 1e-12:
            cleaned.append(float(t))
    return cleaned


def sweep_signal_thresholds(
    pred_ret: np.ndarray,
    exit_ret_arr: np.ndarray,
    cfg: Dict[str, Any],
    min_trades: int,
    target_trade_rate: Optional[float],
) -> pd.DataFrame:
    grid = build_signal_threshold_grid(
        pred_ret=pred_ret,
        base_grid=cfg.get("signal_thr_grid", [0.0]),
        target_trades_list=cfg.get("proxy_target_trades", None),
    )

    obj = str(cfg.get("pnl_objective", "pnl_sum"))
    max_rate = cfg.get("max_trade_rate_val", None)
    penalty = float(cfg.get("trade_rate_penalty", 0.0))

    rows = []
    for thr in grid:
        m = pnl_from_ret_forecast(pred_ret, exit_ret_arr, thr, cfg["cost_bps"])
        if int(m["n_trades"]) < int(min_trades):
            continue
        if max_rate is not None and float(m["trade_rate"]) > float(max_rate):
            continue

        base = float(m.get(obj, np.nan))
        if not np.isfinite(base):
            continue

        if target_trade_rate is not None:
            score = base - penalty * abs(float(m["trade_rate"]) - float(target_trade_rate))
        else:
            score = base - penalty * float(m["trade_rate"])

        rows.append({"signal_thr": float(thr), "score": float(score), **m})

    if not rows:
        for thr in grid:
            m = pnl_from_ret_forecast(pred_ret, exit_ret_arr, thr, cfg["cost_bps"])
            base = float(m.get(obj, np.nan))
            if np.isfinite(base):
                rows.append({"signal_thr": float(thr), "score": float(base), **m})

    if not rows:
        return pd.DataFrame(columns=["signal_thr", "score", "n", "n_trades", "trade_rate", "pnl_sum", "pnl_mean", "pnl_per_trade"])

    return pd.DataFrame(rows).sort_values(["score", "pnl_sum"], ascending=False)

In [9]:
def _to_jsonable_cfg(cfg: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in cfg.items():
        out[k] = str(v) if isinstance(v, Path) else v
    return out


def save_scaler_npz(path: Path, params: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        str(path),
        center_=np.asarray(params["center_"], dtype=np.float32),
        scale_=np.asarray(params["scale_"], dtype=np.float32),
        max_abs=np.asarray([float(params["max_abs"])], dtype=np.float32),
    )


def load_scaler_npz(path: Path) -> Dict[str, Any]:
    data = np.load(str(path))
    return {
        "center_": data["center_"].astype(np.float32),
        "scale_": data["scale_"].astype(np.float32),
        "max_abs": float(data["max_abs"][0]),
    }


def save_bundle(
    bundle_dir: Path,
    name: str,
    model_state: Dict[str, torch.Tensor],
    cfg: Dict[str, Any],
    node_scaler_params: Dict[str, Any],
    edge_scaler_params: Optional[Dict[str, Any]],
    extra_meta: Dict[str, Any],
) -> Dict[str, Path]:
    bundle_dir.mkdir(parents=True, exist_ok=True)
    weights_path = bundle_dir / f"{name}_weights.pt"
    node_scaler_path = bundle_dir / f"{name}_node_scaler.npz"
    edge_scaler_path = bundle_dir / f"{name}_edge_scaler.npz"
    meta_path = bundle_dir / f"{name}_meta.json"

    torch.save(model_state, str(weights_path))
    save_scaler_npz(node_scaler_path, node_scaler_params)

    edge_scaler_file = None
    if edge_scaler_params is not None:
        save_scaler_npz(edge_scaler_path, edge_scaler_params)
        edge_scaler_file = edge_scaler_path.name

    meta = {
        "name": name,
        "weights_file": weights_path.name,
        "node_scaler_file": node_scaler_path.name,
        "edge_scaler_file": edge_scaler_file,
        "cfg": _to_jsonable_cfg(cfg),
        **extra_meta,
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {
        "weights": weights_path,
        "node_scaler": node_scaler_path,
        "edge_scaler": edge_scaler_path if edge_scaler_params is not None else None,
        "meta": meta_path,
    }


def load_bundle(bundle_dir: Path, name: str) -> Dict[str, Any]:
    meta_path = bundle_dir / f"{name}_meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(str(meta_path))

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    weights_path = bundle_dir / meta["weights_file"]
    node_scaler_path = bundle_dir / meta["node_scaler_file"]
    edge_scaler_file = meta.get("edge_scaler_file", None)
    edge_scaler_path = (bundle_dir / edge_scaler_file) if edge_scaler_file else None

    state = torch.load(str(weights_path), map_location="cpu")
    node_scaler_params = load_scaler_npz(node_scaler_path)
    edge_scaler_params = load_scaler_npz(edge_scaler_path) if edge_scaler_path else None

    return {
        "meta": meta,
        "state": state,
        "node_scaler_params": node_scaler_params,
        "edge_scaler_params": edge_scaler_params,
    }

# Legacy Block Removed

The original multi-head classification architecture is replaced by the regression-first workflow defined above.

In [10]:
# Legacy architecture block removed in this rewrite.
# The active model is AttentionGraphPriceModel from Cell 16.

# Legacy Loss Block Removed

Trading-classification losses are not used in this rewrite.

In [11]:
@torch.no_grad()
def eval_price_on_indices(
    model: nn.Module,
    x_scaled: np.ndarray,
    edge_scaled: np.ndarray,
    indices: np.ndarray,
    cfg: Dict[str, Any],
) -> Dict[str, Any]:
    ds = LobGraphSequenceDatasetPrice(
        x_node=x_scaled,
        e_feat=edge_scaled,
        y_ret_arr=ret_target,
        exit_ret_arr=exit_ret,
        sample_t_=sample_t,
        indices=indices.astype(np.int64),
        lookback=int(cfg["lookback"]),
    )
    loader = DataLoader(ds, batch_size=int(cfg["batch_size"]), shuffle=False, collate_fn=collate_fn_price, num_workers=0)

    model.eval()

    tot_loss = 0.0
    n = 0

    pred_all = []
    yret_all = []
    exit_all = []

    for x, e_last, yret, er_exit, _sidx in loader:
        x = x.to(DEVICE).float()
        e_last = e_last.to(DEVICE).float()
        yret = yret.to(DEVICE).float()

        ret_hat, aux = model(x, e_last, return_aux=True)
        loss = regression_loss(ret_hat, yret, cfg)
        loss = total_loss_with_adj_reg(loss, aux, cfg)

        bsz = int(yret.size(0))
        tot_loss += float(loss.item()) * bsz
        n += bsz

        pred_all.append(ret_hat.detach().cpu().numpy())
        yret_all.append(yret.detach().cpu().numpy())
        exit_all.append(er_exit.detach().cpu().numpy())

    pred_np = np.concatenate(pred_all, axis=0) if pred_all else np.zeros((0,), dtype=np.float64)
    yret_np = np.concatenate(yret_all, axis=0) if yret_all else np.zeros((0,), dtype=np.float64)
    exit_np = np.concatenate(exit_all, axis=0) if exit_all else np.zeros((0,), dtype=np.float64)

    return {
        "loss": float(tot_loss / max(1, n)),
        "rmse": rmse_np(yret_np, pred_np),
        "mae": mae_np(yret_np, pred_np),
        "pred_ret": pred_np,
        "true_ret": yret_np,
        "exit_ret": exit_np,
    }


def _proxy_trade_rate(indices: np.ndarray, sample_t_: np.ndarray, exit_ret_arr: np.ndarray, cost_bps: float) -> float:
    t_idx = sample_t_[indices.astype(np.int64)]
    arr = np.asarray(exit_ret_arr[t_idx], dtype=np.float64)
    fee = float(cost_bps) * 1e-4
    return float((np.abs(arr) > fee).mean()) if arr.size else float("nan")


def train_one_fold_price(
    fold_id: int,
    x_scaled: np.ndarray,
    edge_scaled: np.ndarray,
    idx_train: np.ndarray,
    idx_val: np.ndarray,
    idx_test: np.ndarray,
    node_scaler_params: Dict[str, Any],
    edge_scaler_params: Optional[Dict[str, Any]],
    cfg: Dict[str, Any],
) -> Dict[str, Any]:
    tr_ds = LobGraphSequenceDatasetPrice(x_scaled, edge_scaled, ret_target, exit_ret, sample_t, idx_train, int(cfg["lookback"]))
    va_ds = LobGraphSequenceDatasetPrice(x_scaled, edge_scaled, ret_target, exit_ret, sample_t, idx_val, int(cfg["lookback"]))

    tr_loader = DataLoader(tr_ds, batch_size=int(cfg["batch_size"]), shuffle=True, collate_fn=collate_fn_price, num_workers=0)
    va_loader = DataLoader(va_ds, batch_size=int(cfg["batch_size"]), shuffle=False, collate_fn=collate_fn_price, num_workers=0)

    model = AttentionGraphPriceModel(
        node_in=int(x_scaled.shape[-1]),
        cfg=cfg,
        n_nodes=len(ASSETS),
        target_node=TARGET_NODE,
    ).to(DEVICE)

    opt = torch.optim.AdamW(model.parameters(), lr=float(cfg["lr"]), weight_decay=float(cfg["weight_decay"]))

    use_onecycle = bool(cfg.get("use_onecycle", True))
    if use_onecycle:
        sch = torch.optim.lr_scheduler.OneCycleLR(
            opt,
            max_lr=float(cfg["lr"]),
            epochs=int(cfg["epochs"]),
            steps_per_epoch=max(1, len(tr_loader)),
            pct_start=float(cfg.get("onecycle_pct_start", 0.20)),
            div_factor=float(cfg.get("onecycle_div_factor", 25.0)),
            final_div_factor=float(cfg.get("onecycle_final_div", 200.0)),
        )
    else:
        sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="min", factor=0.5, patience=3)

    best_sel = float("inf")
    best_state = None
    best_epoch = -1
    patience = 7
    bad = 0

    for ep in range(1, int(cfg["epochs"]) + 1):
        model.train()
        tr_tot = 0.0
        tr_n = 0

        for x, e_last, yret, _er_exit, _sidx in tr_loader:
            x = x.to(DEVICE).float()
            e_last = e_last.to(DEVICE).float()
            yret = yret.to(DEVICE).float()

            opt.zero_grad(set_to_none=True)
            ret_hat, aux = model(x, e_last, return_aux=True)
            loss = regression_loss(ret_hat, yret, cfg)
            loss = total_loss_with_adj_reg(loss, aux, cfg)

            if not torch.isfinite(loss):
                continue

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), float(cfg["grad_clip"]))
            opt.step()
            if use_onecycle:
                sch.step()

            bsz = int(yret.size(0))
            tr_tot += float(loss.item()) * bsz
            tr_n += bsz

        tr_loss = tr_tot / max(1, tr_n)

        model.eval()
        va_tot = 0.0
        va_n = 0
        va_pred = []
        va_true = []

        with torch.no_grad():
            for x, e_last, yret, _er_exit, _sidx in va_loader:
                x = x.to(DEVICE).float()
                e_last = e_last.to(DEVICE).float()
                yret = yret.to(DEVICE).float()

                ret_hat, aux = model(x, e_last, return_aux=True)
                loss = regression_loss(ret_hat, yret, cfg)
                loss = total_loss_with_adj_reg(loss, aux, cfg)

                bsz = int(yret.size(0))
                va_tot += float(loss.item()) * bsz
                va_n += bsz

                va_pred.append(ret_hat.detach().cpu().numpy())
                va_true.append(yret.detach().cpu().numpy())

        va_pred_np = np.concatenate(va_pred, axis=0) if va_pred else np.zeros((0,), dtype=np.float64)
        va_true_np = np.concatenate(va_true, axis=0) if va_true else np.zeros((0,), dtype=np.float64)
        va_rmse = rmse_np(va_true_np, va_pred_np)
        va_mae = mae_np(va_true_np, va_pred_np)
        va_loss = va_tot / max(1, va_n)

        sel = float(va_rmse)

        if sel < best_sel:
            best_sel = sel
            best_epoch = ep
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad = 0
        else:
            bad += 1

        if not use_onecycle:
            sch.step(sel)

        lr_now = opt.param_groups[0]["lr"]
        w_support = model.support_mix().detach().cpu().numpy().tolist()
        print(
            f"[fold {fold_id:02d}] ep {ep:02d} lr={lr_now:.2e} "
            f"tr_loss={tr_loss:.6f} val_loss={va_loss:.6f} val_rmse={va_rmse:.6f} val_mae={va_mae:.6f} "
            f"best_rmse={best_sel:.6f}@ep{best_epoch:02d} supports={np.round(w_support, 3).tolist()}"
        )

        if bad >= patience:
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    val_eval = eval_price_on_indices(model, x_scaled, edge_scaled, idx_val, cfg)
    test_eval = eval_price_on_indices(model, x_scaled, edge_scaled, idx_test, cfg)

    target_rate = _proxy_trade_rate(idx_val, sample_t, exit_ret, cfg["cost_bps"])
    sweep_val = sweep_signal_thresholds(
        pred_ret=val_eval["pred_ret"],
        exit_ret_arr=val_eval["exit_ret"],
        cfg=cfg,
        min_trades=int(cfg["eval_min_trades"]),
        target_trade_rate=target_rate,
    )
    if len(sweep_val) == 0:
        raise RuntimeError("Threshold sweep returned no valid rows on validation set.")

    best_thr = sweep_val.iloc[0].to_dict()
    signal_thr = float(best_thr["signal_thr"])

    pnl_val = pnl_from_ret_forecast(val_eval["pred_ret"], val_eval["exit_ret"], signal_thr, cfg["cost_bps"])
    pnl_test = pnl_from_ret_forecast(test_eval["pred_ret"], test_eval["exit_ret"], signal_thr, cfg["cost_bps"])

    print(
        f"[fold {fold_id:02d}] selected signal_thr={signal_thr:.6f} "
        f"| val pnl_sum={pnl_val['pnl_sum']:.4f} val trade_rate={pnl_val['trade_rate']:.3f}"
    )
    print(
        f"[fold {fold_id:02d}] TEST: rmse={test_eval['rmse']:.6f} mae={test_eval['mae']:.6f} "
        f"pnl_sum={pnl_test['pnl_sum']:.4f} trade_rate={pnl_test['trade_rate']:.3f} trades={pnl_test['n_trades']}"
    )

    return {
        "fold": int(fold_id),
        "model_state": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
        "node_scaler_params": node_scaler_params,
        "edge_scaler_params": edge_scaler_params,
        "idx_train": idx_train,
        "idx_val": idx_val,
        "idx_test": idx_test,
        "best_epoch": int(best_epoch),
        "best_sel": float(best_sel),
        "val_eval": val_eval,
        "test_eval": test_eval,
        "signal_thr": signal_thr,
        "pnl_val": pnl_val,
        "pnl_test": pnl_test,
    }


def run_walk_forward_cv_price() -> Tuple[pd.DataFrame, List[Dict[str, Any]], str]:
    fold_artifacts: List[Dict[str, Any]] = []
    rows: List[Dict[str, Any]] = []

    best_overall_sel = float("inf")
    best_overall_name = None

    for fi, (idx_tr, idx_va, idx_te) in enumerate(walk_splits, 1):
        print("\n" + "=" * 90)
        print(f"FOLD {fi}/{len(walk_splits)} sizes: train={len(idx_tr)} val={len(idx_va)} test={len(idx_te)}")

        x_scaled, node_params = fit_scale_nodes_train_only(X_node_raw, sample_t, idx_tr, max_abs=float(CFG["max_abs_feat"]))
        if bool(CFG.get("edge_scale", True)):
            edge_scaled, edge_params = fit_scale_edges_train_only(edge_feat, sample_t, idx_tr, max_abs=float(CFG["max_abs_edge"]))
        else:
            edge_scaled = edge_feat.astype(np.float32)
            edge_params = None

        artifact = train_one_fold_price(
            fold_id=fi,
            x_scaled=x_scaled,
            edge_scaled=edge_scaled,
            idx_train=idx_tr,
            idx_val=idx_va,
            idx_test=idx_te,
            node_scaler_params=node_params,
            edge_scaler_params=edge_params,
            cfg=CFG,
        )

        fold_name = f"fold_{fi:02d}_price"
        extra_meta = {
            "kind": "fold_best",
            "fold": fi,
            "best_epoch": artifact["best_epoch"],
            "best_sel": artifact["best_sel"],
            "signal_thr": artifact["signal_thr"],
            "idx_train": artifact["idx_train"].tolist(),
            "idx_val": artifact["idx_val"].tolist(),
            "idx_test": artifact["idx_test"].tolist(),
        }
        save_bundle(
            bundle_dir=ART_DIR,
            name=fold_name,
            model_state=artifact["model_state"],
            cfg=CFG,
            node_scaler_params=artifact["node_scaler_params"],
            edge_scaler_params=artifact["edge_scaler_params"],
            extra_meta=extra_meta,
        )

        if float(artifact["best_sel"]) < best_overall_sel:
            best_overall_sel = float(artifact["best_sel"])
            best_overall_name = fold_name

        fold_artifacts.append(artifact)
        rows.append({
            "fold": fi,
            "val_rmse": artifact["val_eval"]["rmse"],
            "val_mae": artifact["val_eval"]["mae"],
            "test_rmse": artifact["test_eval"]["rmse"],
            "test_mae": artifact["test_eval"]["mae"],
            "signal_thr": artifact["signal_thr"],
            "test_trade_rate": artifact["pnl_test"]["trade_rate"],
            "test_pnl_sum": artifact["pnl_test"]["pnl_sum"],
            "test_n_trades": artifact["pnl_test"]["n_trades"],
            "best_sel_rmse": artifact["best_sel"],
        })

    cv_summary = pd.DataFrame(rows)
    assert best_overall_name is not None

    overall_name = "overall_best_price"
    best_bundle = load_bundle(ART_DIR, best_overall_name)
    extra_meta = {
        "kind": "overall_best",
        "source_name": best_overall_name,
        "source_fold": best_bundle["meta"].get("fold", None),
        "signal_thr": best_bundle["meta"]["signal_thr"],
        "idx_train": best_bundle["meta"]["idx_train"],
        "idx_val": best_bundle["meta"]["idx_val"],
        "idx_test": best_bundle["meta"]["idx_test"],
    }
    save_bundle(
        bundle_dir=ART_DIR,
        name=overall_name,
        model_state=best_bundle["state"],
        cfg=CFG,
        node_scaler_params=best_bundle["node_scaler_params"],
        edge_scaler_params=best_bundle["edge_scaler_params"],
        extra_meta=extra_meta,
    )

    print("\nSaved overall best bundle:", overall_name)
    return cv_summary, fold_artifacts, overall_name


cv_summary_price, fold_artifacts_price, overall_best_name = run_walk_forward_cv_price()

print("\n" + "=" * 90)
print("CV summary:")
print(cv_summary_price)
print("\nCV means:")
print(cv_summary_price.mean(numeric_only=True))


FOLD 1/3 sizes: train=3955 val=719 test=719


/opt/miniconda3/envs/recbole_new_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[fold 01] ep 01 lr=1.43e-05 tr_loss=0.001709 val_loss=0.001218 val_rmse=0.412670 val_mae=0.323043 best_rmse=0.412670@ep01 supports=[0.333, 0.333, 0.333]
[fold 01] ep 02 lr=3.15e-05 tr_loss=0.001631 val_loss=0.001163 val_rmse=0.357585 val_mae=0.286223 best_rmse=0.357585@ep02 supports=[0.333, 0.334, 0.333]
[fold 01] ep 03 lr=5.83e-05 tr_loss=0.001472 val_loss=0.000949 val_rmse=0.176474 val_mae=0.143990 best_rmse=0.176474@ep03 supports=[0.333, 0.334, 0.333]
[fold 01] ep 04 lr=9.21e-05 tr_loss=0.001238 val_loss=0.000837 val_rmse=0.086425 val_mae=0.069627 best_rmse=0.086425@ep04 supports=[0.333, 0.335, 0.333]
[fold 01] ep 05 lr=1.29e-04 tr_loss=0.001111 val_loss=0.000804 val_rmse=0.062400 val_mae=0.049063 best_rmse=0.062400@ep05 supports=[0.332, 0.336, 0.332]
[fold 01] ep 06 lr=1.67e-04 tr_loss=0.000991 val_loss=0.000772 val_rmse=0.038697 val_mae=0.031199 best_rmse=0.038697@ep06 supports=[0.332, 0.337, 0.332]
[fold 01] ep 07 lr=2.01e-04 tr_loss=0.000919 val_loss=0.000748 val_rmse=0.027806 v

/opt/miniconda3/envs/recbole_new_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[fold 02] ep 01 lr=1.43e-05 tr_loss=0.001955 val_loss=0.001578 val_rmse=0.691493 val_mae=0.561372 best_rmse=0.691493@ep01 supports=[0.333, 0.334, 0.333]
[fold 02] ep 02 lr=3.15e-05 tr_loss=0.001723 val_loss=0.001241 val_rmse=0.416986 val_mae=0.336928 best_rmse=0.416986@ep02 supports=[0.333, 0.334, 0.333]
[fold 02] ep 03 lr=5.83e-05 tr_loss=0.001452 val_loss=0.001082 val_rmse=0.267673 val_mae=0.230977 best_rmse=0.267673@ep03 supports=[0.333, 0.334, 0.333]
[fold 02] ep 04 lr=9.20e-05 tr_loss=0.001206 val_loss=0.000922 val_rmse=0.143080 val_mae=0.125059 best_rmse=0.143080@ep04 supports=[0.333, 0.335, 0.333]
[fold 02] ep 05 lr=1.29e-04 tr_loss=0.001068 val_loss=0.000886 val_rmse=0.115660 val_mae=0.102748 best_rmse=0.115660@ep05 supports=[0.332, 0.336, 0.332]
[fold 02] ep 06 lr=1.67e-04 tr_loss=0.000958 val_loss=0.000777 val_rmse=0.043490 val_mae=0.035487 best_rmse=0.043490@ep06 supports=[0.331, 0.337, 0.331]
[fold 02] ep 07 lr=2.00e-04 tr_loss=0.000885 val_loss=0.000771 val_rmse=0.047464 v

/opt/miniconda3/envs/recbole_new_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[fold 03] ep 01 lr=1.43e-05 tr_loss=0.002243 val_loss=0.001414 val_rmse=0.565001 val_mae=0.461945 best_rmse=0.565001@ep01 supports=[0.333, 0.333, 0.333]
[fold 03] ep 02 lr=3.15e-05 tr_loss=0.001932 val_loss=0.001194 val_rmse=0.402720 val_mae=0.315623 best_rmse=0.402720@ep02 supports=[0.333, 0.333, 0.333]
[fold 03] ep 03 lr=5.83e-05 tr_loss=0.001619 val_loss=0.000986 val_rmse=0.238230 val_mae=0.177180 best_rmse=0.238230@ep03 supports=[0.333, 0.334, 0.333]
[fold 03] ep 04 lr=9.20e-05 tr_loss=0.001261 val_loss=0.000844 val_rmse=0.113659 val_mae=0.084736 best_rmse=0.113659@ep04 supports=[0.333, 0.334, 0.333]
[fold 03] ep 05 lr=1.29e-04 tr_loss=0.001055 val_loss=0.000793 val_rmse=0.067829 val_mae=0.052693 best_rmse=0.067829@ep05 supports=[0.332, 0.335, 0.332]
[fold 03] ep 06 lr=1.67e-04 tr_loss=0.000949 val_loss=0.000766 val_rmse=0.052789 val_mae=0.040861 best_rmse=0.052789@ep06 supports=[0.332, 0.336, 0.332]
[fold 03] ep 07 lr=2.00e-04 tr_loss=0.000872 val_loss=0.000721 val_rmse=0.033419 v

# Legacy Threshold Check Removed

Threshold sweep for the new workflow is implemented by sweep_signal_thresholds in Cell 16.

In [12]:
# Legacy threshold helper block removed.

# Legacy Evaluation Block Removed

Evaluation is performed by eval_price_on_indices in Cell 28.

In [13]:
# Legacy evaluation helper block removed.

 # Step 10: Artifact saving and loading



 This cell saves a training bundle as three file types:

 - model weights in `.pt`,

 - scaler parameters in `.npz`,

 - metadata in `.json`.



 The format is intentionally lightweight and avoids pickle-based artifact

 storage for scalers and metadata.

In [14]:
def _to_jsonable_cfg(cfg: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in cfg.items():
        out[k] = str(v) if isinstance(v, Path) else v
    return out


def save_scaler_npz(path: Path, params: Dict[str, Any]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    np.savez_compressed(
        str(path),
        center_=np.asarray(params["center_"], dtype=np.float32),
        scale_=np.asarray(params["scale_"], dtype=np.float32),
        max_abs=np.asarray([float(params["max_abs"])], dtype=np.float32),
    )


def load_scaler_npz(path: Path) -> Dict[str, Any]:
    data = np.load(str(path))
    return {
        "center_": data["center_"].astype(np.float32),
        "scale_": data["scale_"].astype(np.float32),
        "max_abs": float(data["max_abs"][0]),
    }


def save_bundle(
    bundle_dir: Path,
    name: str,
    model_state: Dict[str, torch.Tensor],
    cfg: Dict[str, Any],
    node_scaler_params: Dict[str, Any],
    edge_scaler_params: Optional[Dict[str, Any]],
    extra_meta: Dict[str, Any],
) -> Dict[str, Path]:
    bundle_dir.mkdir(parents=True, exist_ok=True)
    weights_path = bundle_dir / f"{name}_weights.pt"
    node_scaler_path = bundle_dir / f"{name}_node_scaler.npz"
    edge_scaler_path = bundle_dir / f"{name}_edge_scaler.npz"
    meta_path = bundle_dir / f"{name}_meta.json"

    torch.save(model_state, str(weights_path))
    save_scaler_npz(node_scaler_path, node_scaler_params)

    edge_scaler_file = None
    if edge_scaler_params is not None:
        save_scaler_npz(edge_scaler_path, edge_scaler_params)
        edge_scaler_file = edge_scaler_path.name

    meta = {
        "name": name,
        "weights_file": weights_path.name,
        "node_scaler_file": node_scaler_path.name,
        "edge_scaler_file": edge_scaler_file,
        "cfg": _to_jsonable_cfg(cfg),
        **extra_meta,
    }
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(meta, f, indent=2)

    return {
        "weights": weights_path,
        "node_scaler": node_scaler_path,
        "edge_scaler": edge_scaler_path if edge_scaler_params is not None else None,
        "meta": meta_path,
    }


def load_bundle(bundle_dir: Path, name: str) -> Dict[str, Any]:
    meta_path = bundle_dir / f"{name}_meta.json"
    if not meta_path.exists():
        raise FileNotFoundError(str(meta_path))

    with open(meta_path, "r", encoding="utf-8") as f:
        meta = json.load(f)

    weights_path = bundle_dir / meta["weights_file"]
    node_scaler_path = bundle_dir / meta["node_scaler_file"]
    edge_scaler_file = meta.get("edge_scaler_file", None)
    edge_scaler_path = (bundle_dir / edge_scaler_file) if edge_scaler_file else None

    state = torch.load(str(weights_path), map_location="cpu")
    node_scaler_params = load_scaler_npz(node_scaler_path)
    edge_scaler_params = load_scaler_npz(edge_scaler_path) if edge_scaler_path else None

    return {
        "meta": meta,
        "state": state,
        "node_scaler_params": node_scaler_params,
        "edge_scaler_params": edge_scaler_params,
    }


# Bundle Evaluation on Arbitrary Index Sets

Reload a saved price-regression bundle, apply stored scalers, and evaluate RMSE/MAE and PnL on any index subset.

In [15]:
@torch.no_grad()
def evaluate_bundle_on_indices(
    bundle_dir: Path,
    name: str,
    indices: np.ndarray,
    label: str,
    signal_thr_override: Optional[float] = None,
) -> Dict[str, Any]:
    bundle = load_bundle(bundle_dir, name)
    cfg = bundle["meta"]["cfg"]

    model = AttentionGraphPriceModel(
        node_in=int(X_node_raw.shape[-1]),
        cfg=cfg,
        n_nodes=len(ASSETS),
        target_node=TARGET_NODE,
    ).to(DEVICE)
    model.load_state_dict(bundle["state"])
    model.eval()

    x_scaled = apply_scaler_params(X_node_raw.astype(np.float32), bundle["node_scaler_params"])
    if bundle["edge_scaler_params"] is not None:
        e_scaled = apply_scaler_params(edge_feat.astype(np.float32), bundle["edge_scaler_params"])
    else:
        e_scaled = edge_feat.astype(np.float32)

    ev = eval_price_on_indices(model, x_scaled, e_scaled, indices.astype(np.int64), cfg)

    signal_thr = float(signal_thr_override) if signal_thr_override is not None else float(bundle["meta"]["signal_thr"])
    pnl = pnl_from_ret_forecast(ev["pred_ret"], ev["exit_ret"], signal_thr, float(cfg["cost_bps"]))

    print("\n" + "=" * 90)
    print(label)
    print(f"bundle: {name}")
    print(f"rmse={ev['rmse']:.6f} | mae={ev['mae']:.6f} | loss={ev['loss']:.6f}")
    print(f"signal_thr={signal_thr:.6f} | pnl_sum={pnl['pnl_sum']:.4f} | trade_rate={pnl['trade_rate']:.3f} | trades={pnl['n_trades']}")

    return {"eval": ev, "pnl": pnl, "signal_thr": signal_thr}


holdout_indices = idx_final_test.astype(np.int64)
overall_holdout_eval = evaluate_bundle_on_indices(
    ART_DIR,
    overall_best_name,
    holdout_indices,
    label="HOLDOUT EVALUATION USING overall_best_price",
)

/opt/miniconda3/envs/recbole_new_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(



HOLDOUT EVALUATION USING overall_best_price
bundle: overall_best_price
rmse=0.008984 | mae=0.007269 | loss=0.000428
signal_thr=0.001000 | pnl_sum=-1.4378 | trade_rate=0.790 | trades=631


# Production Fit and Final Holdout PnL

Train on the full CV region with a final validation slice, select threshold on validation only, and report final PnL on the reserved holdout test set.

In [16]:
def production_fit_and_save_price() -> str:
    print("\n" + "=" * 90)
    print("PRODUCTION FIT: train on CV -> select signal threshold on val_final -> evaluate on holdout")

    val_w = max(1, int(float(CFG["val_window_frac"]) * n_samples_cv))
    train_end = n_samples_cv - val_w

    idx_train_final = np.arange(0, train_end, dtype=np.int64)
    idx_val_final = np.arange(train_end, n_samples_cv, dtype=np.int64)
    idx_holdout = idx_final_test.astype(np.int64)

    print("Sizes:")
    print("  train_final:", len(idx_train_final))
    print("  val_final  :", len(idx_val_final))
    print("  holdout    :", len(idx_holdout))

    x_scaled, node_params = fit_scale_nodes_train_only(X_node_raw, sample_t, idx_train_final, max_abs=float(CFG["max_abs_feat"]))
    if bool(CFG.get("edge_scale", True)):
        edge_scaled, edge_params = fit_scale_edges_train_only(edge_feat, sample_t, idx_train_final, max_abs=float(CFG["max_abs_edge"]))
    else:
        edge_scaled = edge_feat.astype(np.float32)
        edge_params = None

    artifact = train_one_fold_price(
        fold_id=99,
        x_scaled=x_scaled,
        edge_scaled=edge_scaled,
        idx_train=idx_train_final,
        idx_val=idx_val_final,
        idx_test=idx_holdout,
        node_scaler_params=node_params,
        edge_scaler_params=edge_params,
        cfg=CFG,
    )

    production_name = "production_best_price"
    extra_meta = {
        "kind": "production_best",
        "fold": 99,
        "best_epoch": artifact["best_epoch"],
        "best_sel": artifact["best_sel"],
        "signal_thr": artifact["signal_thr"],
        "idx_train": artifact["idx_train"].tolist(),
        "idx_val": artifact["idx_val"].tolist(),
        "idx_test": artifact["idx_test"].tolist(),
    }

    save_bundle(
        bundle_dir=ART_DIR,
        name=production_name,
        model_state=artifact["model_state"],
        cfg=CFG,
        node_scaler_params=artifact["node_scaler_params"],
        edge_scaler_params=artifact["edge_scaler_params"],
        extra_meta=extra_meta,
    )

    print("Saved production bundle:", production_name)
    return production_name


production_best_name = production_fit_and_save_price()
production_holdout_eval = evaluate_bundle_on_indices(
    ART_DIR,
    production_best_name,
    idx_final_test.astype(np.int64),
    label="FINAL HOLDOUT TEST FROM production_best_price",
)

print("\n" + "=" * 90)
print("FINAL TEST PNL SUMMARY")
print({
    "model": production_best_name,
    "holdout_rmse": production_holdout_eval["eval"]["rmse"],
    "holdout_mae": production_holdout_eval["eval"]["mae"],
    "holdout_signal_thr": production_holdout_eval["signal_thr"],
    "holdout_trade_rate": production_holdout_eval["pnl"]["trade_rate"],
    "holdout_n_trades": production_holdout_eval["pnl"]["n_trades"],
    "holdout_pnl_sum": production_holdout_eval["pnl"]["pnl_sum"],
    "holdout_pnl_per_trade": production_holdout_eval["pnl"]["pnl_per_trade"],
})


PRODUCTION FIT: train on CV -> select signal threshold on val_final -> evaluate on holdout
Sizes:
  train_final: 6473
  val_final  : 719
  holdout    : 799


/opt/miniconda3/envs/recbole_new_env/lib/python3.11/site-packages/torch/nn/modules/transformer.py:382: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


[fold 99] ep 01 lr=1.43e-05 tr_loss=0.002035 val_loss=0.001220 val_rmse=0.409502 val_mae=0.327073 best_rmse=0.409502@ep01 supports=[0.333, 0.334, 0.333]
[fold 99] ep 02 lr=3.15e-05 tr_loss=0.001756 val_loss=0.001061 val_rmse=0.274587 val_mae=0.220611 best_rmse=0.274587@ep02 supports=[0.333, 0.334, 0.333]
[fold 99] ep 03 lr=5.82e-05 tr_loss=0.001512 val_loss=0.000931 val_rmse=0.166322 val_mae=0.134778 best_rmse=0.166322@ep03 supports=[0.333, 0.335, 0.333]
[fold 99] ep 04 lr=9.20e-05 tr_loss=0.001264 val_loss=0.000890 val_rmse=0.122933 val_mae=0.108463 best_rmse=0.122933@ep04 supports=[0.332, 0.336, 0.332]
[fold 99] ep 05 lr=1.29e-04 tr_loss=0.001059 val_loss=0.000782 val_rmse=0.047983 val_mae=0.040693 best_rmse=0.047983@ep05 supports=[0.331, 0.338, 0.331]
[fold 99] ep 06 lr=1.67e-04 tr_loss=0.000938 val_loss=0.000808 val_rmse=0.072002 val_mae=0.067858 best_rmse=0.047983@ep05 supports=[0.33, 0.34, 0.33]
[fold 99] ep 07 lr=2.00e-04 tr_loss=0.000841 val_loss=0.000733 val_rmse=0.040415 val_

# Optional Diagnostics

Use this cell for additional custom checks if needed.

In [17]:
print("Diagnostics cell: no extra checks by default.")

Diagnostics cell: no extra checks by default.


# Legacy Section Removed

The old bundle evaluation logic is replaced by evaluate_bundle_on_indices above.

In [18]:
print("Legacy block removed.")

Legacy block removed.


# End of Notebook

The active workflow has already produced final holdout RMSE/MAE and PnL.

In [19]:
print("Notebook rewrite complete.")

Notebook rewrite complete.
